In [ ]:
%pip install -q -U "transformers>=4.57.3,<5" "qwen-vl-utils>=0.0.14"

import os
import subprocess
import sys

QWEN_REPO = "/kaggle/working/Qwen3-VL-Embedding"
KERNEL_CACHE = "/kaggle/working/torch_kernels"
os.makedirs(KERNEL_CACHE, exist_ok=True)
os.environ["PYTORCH_KERNEL_CACHE_PATH"] = KERNEL_CACHE

if not os.path.isdir(QWEN_REPO):
    subprocess.run([
        "git", "clone", "--depth", "1",
        "https://github.com/QwenLM/Qwen3-VL-Embedding.git",
        QWEN_REPO,
    ], check=True)
if QWEN_REPO not in sys.path:
    sys.path.insert(0, QWEN_REPO)


In [ ]:
import csv
import glob
import os
from typing import List

import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
from huggingface_hub import snapshot_download
from PIL import Image
from tqdm.auto import tqdm
from src.models.qwen3_vl_embedding import Qwen3VLEmbedder

MODEL_NAME = "Qwen/Qwen3-VL-Embedding-2B"

class QwenDataParallelEncoder(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, input_ids, attention_mask, pixel_values, image_grid_thw, **kwargs):
        pixel_values = pixel_values.flatten(0, 1)
        outputs = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            image_grid_thw=image_grid_thw,
            **kwargs,
        )
        hidden_state = outputs.last_hidden_state
        last_positions = attention_mask.shape[1] - attention_mask.flip(1).argmax(1) - 1
        rows = torch.arange(hidden_state.shape[0], device=hidden_state.device)
        embeddings = hidden_state[rows, last_positions]
        return F.normalize(embeddings.float(), p=2, dim=-1)

@torch.inference_mode()
def encode_batch(frames, embedder, model, device):
    images = [Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)) for frame in frames]
    conversations = [embedder.format_model_input(image=image) for image in images]
    inputs = embedder._preprocess_inputs(conversations)

    pixel_values = inputs["pixel_values"]
    batch_size = len(images)
    if pixel_values.shape[0] % batch_size != 0:
        raise RuntimeError("Images in one batch produced different patch counts.")
    inputs["pixel_values"] = pixel_values.reshape(
        batch_size, -1, *pixel_values.shape[1:]
    )
    inputs = {name: value.to(device) for name, value in inputs.items()}

    try:
        with torch.amp.autocast(device_type="cuda", enabled=device.type == "cuda"):
            return model(**inputs).cpu()
    except torch.cuda.OutOfMemoryError:
        del inputs
        torch.cuda.empty_cache()
        if len(frames) == 1:
            raise
        middle = len(frames) // 2
        return torch.cat([
            encode_batch(frames[:middle], embedder, model, device),
            encode_batch(frames[middle:], embedder, model, device),
        ])

def save_image_webp(img_bgr, path: str, quality: int = 80, resize_factor: float = 0.5):
    if resize_factor != 1.0:
        img_bgr = cv2.resize(img_bgr, (0, 0), fx=resize_factor, fy=resize_factor)
    image = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    os.makedirs(os.path.dirname(path), exist_ok=True)
    image.save(path, format="WEBP", quality=quality)

def save_feature_tensor(tensor: torch.Tensor, path: str):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    torch.save(tensor.cpu(), path)

def process_video(
    video_path: str,
    kf_dir: str,
    maps_dir: str,
    embedder,
    model,
    feature_dir: str,
    device: torch.device,
    sim_threshold: float,
    skip_frames: int,
    batch_size: int,
    resize_factor: float,
    webp_quality: int,
) -> int:
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Failed to open video: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    video_name = os.path.splitext(os.path.basename(video_path))[0]
    map_csv = os.path.join(maps_dir, f"{video_name}_map.csv")
    frames: List = []
    indices: List[int] = []
    previous_feature = None
    keyframe_count = 0

    with open(map_csv, "w", newline="") as file:
        writer = csv.writer(file)
        writer.writerow(["FrameID", "Seconds"])
        progress = tqdm(
            total=total_frames if total_frames > 0 else None,
            desc=f"Processing {video_name}",
            leave=True,
        )

        def flush_batch():
            nonlocal previous_feature, keyframe_count
            if not frames:
                return
            features = encode_batch(frames, embedder, model, device)
            for image, frame_id, feature in zip(frames, indices, features):
                similarity = None if previous_feature is None else torch.dot(previous_feature, feature).item()
                if previous_feature is None or similarity < sim_threshold:
                    filename = f"keyframe_{frame_id}"
                    save_image_webp(
                        image,
                        os.path.join(kf_dir, f"{filename}.webp"),
                        quality=webp_quality,
                        resize_factor=resize_factor,
                    )
                    save_feature_tensor(
                        feature,
                        os.path.join(feature_dir, f"{filename}.pt"),
                    )
                    writer.writerow([frame_id, f"{frame_id / fps:.2f}"])
                    previous_feature = feature
                    keyframe_count += 1
            frames.clear()
            indices.clear()

        frame_id = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_id % (skip_frames + 1) == 0:
                frames.append(frame)
                indices.append(frame_id)
                if len(frames) == batch_size:
                    flush_batch()
            frame_id += 1
            progress.update(1)

        flush_batch()
        progress.close()

    cap.release()
    return keyframe_count

def process_videos(
    video_files,
    output_base,
    sim_threshold=0.95,
    skip_frames=5,
    batch_size=8,
    resize_factor=0.5,
    webp_quality=80,
):
    video_files = sorted(video_files)
    if not video_files:
        raise RuntimeError("No videos found.")

    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    model_path = snapshot_download(MODEL_NAME)
    embedder = Qwen3VLEmbedder(
        model_name_or_path=model_path,
        max_length=1024,
        min_pixels=4 * 32 * 32,
        max_pixels=512 * 512,
        dtype=torch.float16 if device.type == "cuda" else torch.float32,
        attn_implementation="sdpa",
    )
    model = QwenDataParallelEncoder(embedder.model).to(device)
    if torch.cuda.device_count() > 1:
        print(f"Using {torch.cuda.device_count()} GPUs with nn.DataParallel")
        model = nn.DataParallel(model)
    model.eval()

    maps_dir = os.path.join(output_base, "maps")
    os.makedirs(maps_dir, exist_ok=True)
    os.makedirs(os.path.join(output_base, "keyframe"), exist_ok=True)
    os.makedirs(os.path.join(output_base, "embedding"), exist_ok=True)

    for video_path in video_files:
        video_name = os.path.splitext(os.path.basename(video_path))[0]
        kf_dir = os.path.join(output_base, "keyframe", video_name)
        feature_dir = os.path.join(output_base, "embedding", video_name)
        os.makedirs(kf_dir, exist_ok=True)
        os.makedirs(feature_dir, exist_ok=True)
        print(f"Processing video: {video_name}")
        try:
            count = process_video(
                video_path=video_path,
                kf_dir=kf_dir,
                maps_dir=maps_dir,
                embedder=embedder,
                model=model,
                feature_dir=feature_dir,
                device=device,
                sim_threshold=sim_threshold,
                skip_frames=skip_frames,
                batch_size=batch_size,
                resize_factor=resize_factor,
                webp_quality=webp_quality,
            )
            print(f"Total keyframes: {count}")
        except Exception as error:
            print(f"Error processing {video_name}: {error}")
        print("-" * 20)

def process_all_videos(
    input_folder,
    output_base,
    sim_threshold=0.95,
    skip_frames=5,
    batch_size=8,
    resize_factor=0.5,
    webp_quality=80,
    start_index=0,
    pattern="*.mp4",
):
    video_files = sorted(glob.glob(os.path.join(input_folder, pattern)))[start_index:]
    print(f"Found {len(video_files)} videos in {input_folder}")
    process_videos(
        video_files=video_files,
        output_base=output_base,
        sim_threshold=sim_threshold,
        skip_frames=skip_frames,
        batch_size=batch_size,
        resize_factor=resize_factor,
        webp_quality=webp_quality,
    )


In [ ]:
INPUT_FOLDER = "/kaggle/input/datasets/dtdat1725/aic-hcmc-2025-batch-1/video/L21_a"

process_all_videos(
    input_folder=INPUT_FOLDER,
    output_base="/kaggle/working/qwen3vl_data",
    sim_threshold=0.95,
    skip_frames=5,
    batch_size=8,
    resize_factor=0.5,
    webp_quality=80,
    start_index=0,
    pattern="*.mp4",
)


In [ ]:
test_videos = sorted(glob.glob(os.path.join(INPUT_FOLDER, "*.mp4")))
if not test_videos:
    raise RuntimeError(f"No videos found in {INPUT_FOLDER}")

TEST_VIDEO_PATH = test_videos[0]
print(f"Testing one video: {TEST_VIDEO_PATH}")
process_videos(
    video_files=[TEST_VIDEO_PATH],
    output_base="/kaggle/working/qwen3vl_test",
    sim_threshold=0.95,
    skip_frames=5,
    batch_size=8,
    resize_factor=0.5,
    webp_quality=80,
)


In [ ]:
!cd /kaggle/working && zip -rq qwen3vl_results.zip qwen3vl_data
from IPython.display import FileLink
FileLink("/kaggle/working/qwen3vl_results.zip")
